# Agentic Search: Going Beyond RAG

In the foundation module, we built RAG systems that chunk documents, index them, and retrieve relevant chunks. With agents, we can do better.

Traditional RAG has a fundamental limitation. We take a big document, chunk it, and hope the retrieved chunks contain what we need. If we retrieve chunks 2 and 4 out of 5, we're missing information from chunks 1, 3, and 5. We hope to reconstruct the missing context, but it's not always possible.

Think about how you search the web. You don't get pre-chunked fragments. You: 
1. Search for something 
2. Look at the snippets and titles 
3. Open the most relevant articles 
4. Read through them to find what you need

With agents, we can replicate this human approach. Instead of chunking upfront, we let the agent pick which documents to read based on search results.




## Setting Up the Index

In [1]:
uv add "minsearch==0.0.9"

Note: you may need to restart the kernel to use updated packages.


c:\Users\harun\OneDrive\Documents\GitHub\anvil-llm-lab\.venv\Scripts\python.exe: No module named uv


In [3]:
from gitsource import GithubRepositoryDataReader
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(parsed_docs)

# Next, set up our data. Notice that we're NOT chunking documents:


In [6]:
query = 'how do I use evidently to monitor my machine learning models?'

In [7]:
search_results = index.search(query)

## Highlighting Search Results

When we search, we don't want to return the entire document. That's too much context. Instead, we want snippets that help the agent decide which documents to read in full.

Minsearch has a highlighting feature borrowed from Lucene/Elasticsearch:



In [4]:
from minsearch import Highlighter, Tokenizer
from minsearch.tokenizer import DEFAULT_ENGLISH_STOP_WORDS

In [5]:
# Create the highlighter with stop words and stemming:

stopwords = DEFAULT_ENGLISH_STOP_WORDS | {"evidently"}

highlighter = Highlighter(
    highlight_fields=["content"],
    max_matches=3,
    snippet_size=50,
    tokenizer=Tokenizer(stemmer="snowball", stop_words=stopwords)
)

In [8]:
# Test the highlighter:

highlighter.highlight(query, search_results)

# Now we have concise snippets instead of full documents.



[{'title': 'Use HuggingFace models',
  'description': 'How to use models from HuggingFace as evaluators.',
  'content': {'matches': ['...downloading and using ML **models** from HuggingFace. This l...',
    '...criteria from the source **model**, e.g. classify texts by ...',
    '.... There are:\n\n* Ready-to-**use** descriptors that wrap a ...'],
   'total_matches': 33},
  'filename': 'metrics/customize_hf_descriptor.mdx'},
 {'title': 'Evidently Cloud',
  'description': 'How to set up Evidently Cloud account.',
  'icon': 'cloud',
  'content': {'matches': ['... and Evidently Cloud v1, **use** `from evidently.ui.works...'],
   'total_matches': 1},
  'filename': 'docs/setup/cloud.mdx'},
 {'title': 'Why Evidently?',
  'description': 'Why choose Evidently.',
  'content': {'matches': ['...ucts: whether it’s an ML **model**, an LLM app, or a comple...',
    '...workflow.\n\nOur tools are **model**-, framework-, and applic...',
    '.... It’s up to you.\xa0\n- Add **monitoring** as you are re

## File Index

We also need a way to retrieve full documents by filename:

In [9]:
file_index = {doc["filename"]: doc["content"] for doc in parsed_docs}

In [10]:
# Test it:
file_index["docs/setup/self-hosting.mdx"]


'<Note>\n  Evidently Cloud is no longer available as a SaaS product, but you can self-host the open-source Evidently Platform (as described on this page), or use the Evidently library with other tools.\n</Note>\n\nIn addition to using Evidently Python library, you can self-host the UI Service to get a monitoring Dashboard and organize the results of your evaluations. This is optional: you can also view evaluation results in Python or export to JSON or HTML.\n\nTo get a self-hostable Dashboard, you must:\n\n- Create a Workspace (local or remote) to store your data.\n- Launch the UI service.\n\n## 1. Create a Workspace\n\nOnce you [install Evidently](/docs/setup/installation), you will need to create a `workspace`. This designates a remote or local directory where you will store the evaluation results (JSON Reports called `snapshots`), traces or datasets. The UI Service will read the data from this source.\n\nAs a storage backend, Evidently supports:\n\n- a file system\n- any SQL-like da

## Creating the Search Tools

In [ ]:
# Creating the Search Tools

from typing import Any, Dict, List

class SearchTools:
    """
    Provides search and file retrieval utilities over an indexed data store.
    """
    def __init__(self, index: Any, highlighter: Any, file_index: Dict[str, str]) -> None:
        """
        Initialize the SearchTools instance.
        Args:
            index: Searchable index providing a `search` method.
            highlighter: Highlighter used to annotate search results.
            file_index (Dict[str, str]): Mapping of filenames to file contents.
        """
        self.index = index
        self.highlighter = highlighter
        self.file_index = file_index

    def search(self, query: str) -> List[Dict[str, Any]]:
        """
        Search the index for results matching a query and highlight them.
        Args:
            query (str): The search query to look up in the index.
        Returns:
            List[Dict[str, Any]]: A list of highlighted search result objects.
        """
        search_results = self.index.search(query=query, num_results=5)
        return self.highlighter.highlight(query, search_results)

    def get_file(self, filename: str) -> str:
        """
        Retrieve a file's contents by filename.
        Args:
            filename (str): The filename of the file to retrieve.
        Returns:
            str: The file contents if found, otherwise an error message.
        """
        if filename in self.file_index:
            return self.file_index[filename]
        return f"file {filename} does not exist"

# The search method returns highlighted snippets. 
# 
# The get_file method returns the full document content.



In [ ]:
# Create the tools instance:
search_tools = SearchTools(index, highlighter, file_index)


## Agent instructions

The key is designing instructions that guide the agent through a multi-step process:



In [ ]:
instructions = """
You're a documentation assistant.

Answer the user question using only the documentation knowledge base.

Make 3 iterations:

1) First iteration:
   - Perform one search using the search tool to identify potentially relevant documents.
   - Explain (in 2â€“3 sentences) why this search query is appropriate for the user question.

2) Second iteration:
   - Analyze the results from the previous search.
   - Based on the filenames or documents returned, perform:
       - Up to 2 additional search queries to refine or expand coverage, and
       - One or more get_file calls to retrieve the full content of the most relevant documents.
   - For each search or get_file call, explain (in 2â€“3 sentences) why this action is necessary and how it helps answer the question.

3) Third iteration:
   - Analyze the retrieved document contents from get_file.
   - Synthesize the information from these documents into a final answer to the user.

IMPORTANT:
- At every step, explicitly explain your reasoning for each search query or file retrieval.
- Use only facts found in the documentation knowledge base.
- Do not introduce outside knowledge or assumptions.
- If the answer cannot be found in the retrieved documents, clearly inform the user.

Additional notes:
- The knowledge base is entirely about Evidently, so you do not need to include the word "evidently" in search queries.
- Prefer retrieving and analyzing full documents (via get_file) before producing the final answer.
"""


The agent follows a three-iteration pattern: 
1. First iteration: Search to identify relevant documents 
2. Second iteration: Analyze snippets, refine search, and fetch full documents 
3. Third iteration: Combine into a final answer

This mimics how humans research - start broad, dive deeper into relevant sources, then consolidate.



## Running the Agent

In [14]:
# Set up the tools and agent:

from toyaikit.tools import Tools

agent_tools = Tools()
agent_tools.add_tools(search_tools)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'get_file',
  'description': "Retrieve a file's contents by filename.\nArgs:\n    filename (str): The filename of the file to retrieve.\nReturns:\n    str: The file contents if found, otherwise an error message.",
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'filename parameter'}},
   'required': ['filename'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'search',
  'description': 'Search the index for results matching a query and highlight them.\nArgs:\n    query (str): The search query to look up in the index.\nReturns:\n    List[Dict[str, Any]]: A list of highlighted search result objects.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [15]:
# Set up the LLM client and chat interface:

from openai import OpenAI
from toyaikit.llm import OpenAIClient
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

llm_client = OpenAIClient(
    model="gpt-4o-mini",
    client=OpenAI()
)

chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface)

In [16]:
# Create the agent:

agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [17]:
# Run it:

result = agent.loop(
    query + " show me the code",
    callback=runner_callback
)

# The agent searches, analyzes snippets, fetches relevant documents, and writes an answer with code examples.


-> Response received


-> Response received


-> Response received


-> Response received


## Summary

We went from static RAG to agentic search: - No chunking - index full documents - search tool - returns highlighted snippets for decision-making - get_file tool - fetches full documents when needed - Agent picks what to read based on snippets

This is more powerful than traditional RAG because: - No information loss from chunking - Agent can follow its curiosity - More natural research pattern

In the next lesson, we'll translate this to PydanticAI.

